# 5.1 — Clasificacion jerarquica con clusters malignos no supervisados

Replica simplificada del notebook 5.0, usando datos enriquecidos con
etiquetas de cluster (`mal_cluster`) generados en el notebook 1.2.

**Cambio clave**: los 18 tipos de cancer se reemplazan por k clusters malignos
no supervisados. Stage 1 sigue siendo cancer vs no-cancer (sin cambios).
Stage 2 clasifica entre los k clusters malignos en lugar de los 18 tipos.

Ranking:
1. Minimizar `test_cancer_fn`
2. Minimizar `test_cancer_fnr`
3. Maximizar `test_f1_macro`

In [2]:
from __future__ import annotations
from pathlib import Path
import logging
import warnings
from sklearn.exceptions import ConvergenceWarning

import pandas as pd
import numpy as np

from time import perf_counter
from tqdm.auto import tqdm

from genomics_dl.models.train_multiclass import (
    HierarchicalTrainConfig,
    run_hierarchical_training,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
logging.getLogger("alembic").setLevel(logging.ERROR)
logging.getLogger("alembic.runtime.migration").setLevel(logging.ERROR)
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)
warnings.filterwarnings(
    "ignore", category=RuntimeWarning,
    message=".*invalid value encountered in divide.*",
)
warnings.filterwarnings(
    "ignore", category=RuntimeWarning,
    message=".*A worker stopped while some jobs were given to the executor.*",
)

### Rutas y carga de datos (con clusters)

In [3]:
DATA_PROCESSED = Path("../data/processed")
TRAIN_PATH = DATA_PROCESSED / "gse183635_tep_tpm_train_clustered.parquet"
TEST_PATH  = DATA_PROCESSED / "gse183635_tep_tpm_test_clustered.parquet"

df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

df_train.shape, df_test.shape

((1880, 5453), (471, 5453))

### Separacion genes vs metadatos

In [ ]:
gene_cols = [c for c in df_train.columns if str(c).startswith("ENSG")]

assert "mal_cluster" in df_train.columns, "Ejecutar notebook 1.2 primero"
assert "Class_group" in df_train.columns
assert len(gene_cols) > 0

print(f"Genes: {len(gene_cols)}")
print(f"Train: {len(df_train)}, Test: {len(df_test)}")
print(f"\nmal_cluster (train):")
print(df_train["mal_cluster"].value_counts(dropna=False))

## Sweep jerarquico simplificado (8 configuraciones)

In [5]:
def fmt_secs(s: float) -> str:
    s = int(max(0, s))
    h = s // 3600
    m = (s % 3600) // 60
    ss = s % 60
    if h > 0:
        return f"{h:d}h {m:02d}m {ss:02d}s"
    if m > 0:
        return f"{m:d}m {ss:02d}s"
    return f"{ss:d}s"

In [ ]:
BASE_CONFIG = dict(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    class_group_col="Class_group",
    patient_group_col="Patient_group",
    nonmalignant_label="nonMalignant",
    malignant_label="Malignant",
    use_pca=False,
    var_quantile=0.15,
    selector_on_log=False,
    variance_filter_threshold=1e-6,
    cv_splits=5,
    random_state=42,
    experiment_name="gse183635_hierarchical_mal_clustered",
    save_local_bundle=False,
    save_plots=False,
    mlflow_log_artifacts=False,
    mlflow_log_model=False,
    malignant_cluster_col="mal_cluster",  # <-- CLAVE: activa clusters malignos
)

# Grid simplificado basado en mejores configuraciones de 5.0
STAGE1_CLASSIFIERS = [
    ("lightgbm", dict(n_estimators=400, max_depth=6, learning_rate=0.1)),
    ("xgboost", dict(n_estimators=400, max_depth=6, learning_rate=0.1)),
]

STAGE2_CLASSIFIERS = [
    ("lightgbm", dict(n_estimators=400, max_depth=8, learning_rate=0.05)),
    ("xgboost", dict(n_estimators=400, max_depth=8, learning_rate=0.05)),
]

# Diccionarios para modelo final (con mas estimators)
STAGE1_CLASSIFIERS_DICT = {
    "xgboost": dict(n_estimators=650, max_depth=6, learning_rate=0.1),
    "lightgbm": dict(n_estimators=650, max_depth=6, learning_rate=0.1),
}
STAGE2_CLASSIFIERS_DICT = {
    "xgboost": dict(n_estimators=800, max_depth=8, learning_rate=0.05),
    "lightgbm": dict(n_estimators=800, max_depth=8, learning_rate=0.05),
}

STAGE2_WEIGHTINGS = ["sqrt", "balanced"]
STAGE1_MIN_RECALLS = [0.90]

sweep = []
for s1_name, s1_params in STAGE1_CLASSIFIERS:
    for s2_name, s2_params in STAGE2_CLASSIFIERS:
        for s2_weighting in STAGE2_WEIGHTINGS:
            for s1_min_recall in STAGE1_MIN_RECALLS:
                sweep.append(dict(
                    s1_name=s1_name, s1_params=s1_params,
                    s2_name=s2_name, s2_params=s2_params,
                    s2_weighting=s2_weighting,
                    s1_min_recall=s1_min_recall,
                ))

print(f"Total combinaciones: {len(sweep)}")

In [ ]:
results = []
errors = []

total = len(sweep)
start_all = perf_counter()

ema = None
alpha = 0.25
done = 0

pbar = tqdm(sweep, total=total, desc="Sweep hier. mal_clustered", unit="run")

for combo in pbar:
    t0 = perf_counter()

    s1_name = combo["s1_name"]
    s1_params = combo["s1_params"]
    s2_name = combo["s2_name"]
    s2_params = combo["s2_params"]
    s2_weighting = combo["s2_weighting"]
    s1_min_recall = combo["s1_min_recall"]

    model_name = f"hier_malclust_{s1_name}_{s2_name}"
    model_version = f"w{s2_weighting}_r{int(s1_min_recall*100)}"

    cfg = HierarchicalTrainConfig(
        **BASE_CONFIG,
        model_name=model_name,
        model_version=model_version,
        stage1_clf_name=s1_name,
        stage1_clf_params=s1_params,
        stage1_min_recall=s1_min_recall,
        stage2_clf_name=s2_name,
        stage2_clf_params=s2_params,
        stage2_class_weighting=s2_weighting,
    )

    try:
        res = run_hierarchical_training(cfg, feature_cols=gene_cols)
        tm = res["test_metrics"]
        results.append({
            "model_name": model_name,
            "model_version": model_version,
            "stage1_clf": s1_name,
            "stage2_clf": s2_name,
            "stage2_weighting": s2_weighting,
            "stage1_min_recall": s1_min_recall,
            "test_cancer_fn": tm["cancer_fn"],
            "test_cancer_fnr": tm["cancer_fnr"],
            "test_cancer_recall": tm["cancer_recall_sensitivity"],
            "test_cancer_precision": tm["cancer_precision"],
            "test_cancer_specificity": tm["cancer_specificity"],
            "test_f1_macro": tm["f1_macro"],
            "test_f1_weighted": tm["f1_weighted"],
            "test_accuracy": tm["accuracy"],
            "test_balanced_accuracy": tm["balanced_accuracy"],
            "test_cancer_roc_auc": tm["cancer_roc_auc"],
            "test_cancer_pr_auc": tm["cancer_pr_auc"],
            "threshold": res["chosen_cancer_threshold"],
            "mlflow_run_id": res["mlflow_run_id"],
            "error": None,
        })
    except Exception as e:
        errors.append({
            "model_name": model_name,
            "model_version": model_version,
            "stage1_clf": s1_name,
            "stage2_clf": s2_name,
            "stage2_weighting": s2_weighting,
            "error": repr(e),
        })

    dt = perf_counter() - t0
    ema = dt if ema is None else (alpha * dt + (1 - alpha) * ema)

    done += 1
    elapsed = perf_counter() - start_all
    remaining = (total - done) * (ema if ema is not None else 0.0)

    pbar.set_postfix({
        "last": fmt_secs(dt), "avg": fmt_secs(ema),
        "elapsed": fmt_secs(elapsed), "eta": fmt_secs(remaining),
        "ok": len(results), "err": len(errors),
    })

res_df = (
    pd.DataFrame(results)
      .sort_values(["test_cancer_fn", "test_cancer_fnr", "test_f1_macro"],
                   ascending=[True, True, False])
      .reset_index(drop=True)
)

err_df = pd.DataFrame(errors).reset_index(drop=True)

print(f"OK: {len(res_df)}, Errores: {len(err_df)}")

In [8]:
display(res_df)

,model_name,model_version,stage1_clf,stage2_clf,stage2_weighting,stage1_min_recall,test_cancer_fn,test_cancer_fnr,test_cancer_recall,test_cancer_precision,test_cancer_specificity,test_f1_macro,test_f1_weighted,test_accuracy,test_balanced_accuracy,test_cancer_roc_auc,test_cancer_pr_auc,threshold,mlflow_run_id,error
0,hier_clust_lightgbm_lightgbm,wsqrt_r90,lightgbm,lightgbm,sqrt,0.9,33,0.101227,0.898773,0.851744,0.648276,0.326147,0.389263,0.441614,0.322119,0.873704,0.934113,0.524944,011a2985ae0e4e95a834a3d1ebd78aeb,None
1,hier_clust_lightgbm_lightgbm,wbalanced_r90,lightgbm,lightgbm,balanced,0.9,33,0.101227,0.898773,0.851744,0.648276,0.317404,0.383210,0.433121,0.318517,0.873704,0.934113,0.524944,5bb5bb48dc2c4dffa9ee8c853060c97a,None
2,hier_clust_lightgbm_xgboost,wsqrt_r90,lightgbm,xgboost,sqrt,0.9,33,0.101227,0.898773,0.851744,0.648276,0.307827,0.372394,0.430998,0.300701,0.873704,0.934113,0.524944,f3dabd1ada3f4277a4af20113a5efa10,None
3,hier_clust_lightgbm_xgboost,wbalanced_r90,lightgbm,xgboost,balanced,0.9,33,0.101227,0.898773,0.851744,0.648276,0.301957,0.374519,0.420382,0.307050,0.873704,0.934113,0.524944,f2669ba86ec440c6a56b2d2ba88c250d,None
4,hier_clust_xgboost_lightgbm,wsqrt_r90,xgboost,lightgbm,sqrt,0.9,35,0.107362,0.892638,0.848397,0.641379,0.332052,0.394557,0.447983,0.326463,0.876010,0.938155,0.429169,4e3433defb224d918dcc260f626b82bb,None
5,hier_clust_xgboost_lightgbm,wbalanced_r90,xgboost,lightgbm,balanced,0.9,35,0.107362,0.892638,0.848397,0.641379,0.323788,0.389876,0.441614,0.323337,0.876010,0.938155,0.429169,ecfbc38a56254b4c8a9878a2b29ee25a,None
6,hier_clust_xgboost_xgboost,wsqrt_r90,xgboost,xgboost,sqrt,0.9,35,0.107362,0.892638,0.848397,0.641379,0.308992,0.375689,0.435244,0.302413,0.876010,0.938155,0.429169,f345805a659441e19a0b32d9a89b638d,None
7,hier_clust_xgboost_xgboost,wbalanced_r90,xgboost,xgboost,balanced,0.9,35,0.107362,0.892638,0.848397,0.641379,0.304072,0.377865,0.424628,0.308762,0.876010,0.938155,0.429169,741af9cadd6d406ea7f190f025f29386,None


In [9]:
if len(err_df) > 0:
    display(err_df)

## Entrenamiento final del mejor modelo

In [10]:
best = res_df.iloc[0].to_dict()
print("Mejor configuracion:")
for k, v in best.items():
    if k not in ["mlflow_run_id", "error"]:
        print(f"  {k}: {v}")

Mejor configuracion:
  model_name: hier_clust_lightgbm_lightgbm
  model_version: wsqrt_r90
  stage1_clf: lightgbm
  stage2_clf: lightgbm
  stage2_weighting: sqrt
  stage1_min_recall: 0.9
  test_cancer_fn: 33
  test_cancer_fnr: 0.10122699386503067
  test_cancer_recall: 0.8987730061349694
  test_cancer_precision: 0.8517441860465116
  test_cancer_specificity: 0.6482758620689655
  test_f1_macro: 0.3261472293566613
  test_f1_weighted: 0.3892630522514685
  test_accuracy: 0.4416135881104034
  test_balanced_accuracy: 0.3221193593882939
  test_cancer_roc_auc: 0.8737042521683943
  test_cancer_pr_auc: 0.9341130846393169
  threshold: 0.5249443025698323


In [ ]:
BEST_CONFIG = HierarchicalTrainConfig(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    class_group_col="Class_group",
    patient_group_col="Patient_group",
    nonmalignant_label="nonMalignant",
    malignant_label="Malignant",
    use_pca=False,
    var_quantile=0.15,
    selector_on_log=False,
    variance_filter_threshold=1e-6,
    cv_splits=10,  # CV completo para modelo final
    random_state=42,
    experiment_name="gse183635_hierarchical_mal_clustered",
    model_name="hierarchical_mal_clustered_final",
    model_version="v0.1.0",
    stage1_clf_name=best["stage1_clf"],
    stage1_clf_params=STAGE1_CLASSIFIERS_DICT[best["stage1_clf"]],
    stage1_min_recall=best["stage1_min_recall"],
    stage2_clf_name=best["stage2_clf"],
    stage2_clf_params=STAGE2_CLASSIFIERS_DICT[best["stage2_clf"]],
    stage2_class_weighting=best["stage2_weighting"],
    save_local_bundle=True,
    save_plots=True,
    mlflow_log_artifacts=True,
    mlflow_log_model=True,
    output_models_dir="models",
    output_figures_dir="reports/figures/hierarchical_mal_clustered",
    malignant_cluster_col="mal_cluster",  # activa clusters malignos
)

print("Entrenando modelo final con CV completo...")
final_result = run_hierarchical_training(BEST_CONFIG, feature_cols=gene_cols)
print(f"\nModelo guardado en: {final_result.get('bundle_dir', 'N/A')}")

## Metricas del modelo final

In [12]:
tm = final_result["test_metrics"]

print("METRICAS MODELO FINAL")
print("\n--- Metricas de Test ---")
print(f"  Accuracy:           {tm['accuracy']:.4f}")
print(f"  Balanced Accuracy:  {tm['balanced_accuracy']:.4f}")
print(f"  F1-macro:           {tm['f1_macro']:.4f}")
print(f"  F1-weighted:        {tm['f1_weighted']:.4f}")

print("\n--- Deteccion de Cancer (Etapa 1) ---")
print(f"  Umbral:             {final_result['chosen_cancer_threshold']:.4f}")
print(f"  Recall (Sens.):     {tm['cancer_recall_sensitivity']:.4f}")
print(f"  Especificidad:      {tm['cancer_specificity']:.4f}")
print(f"  Precision:          {tm['cancer_precision']:.4f}")
print(f"  FNR:                {tm['cancer_fnr']:.4f}")
print(f"  TP: {tm['cancer_tp']}, FP: {tm['cancer_fp']}, TN: {tm['cancer_tn']}, FN: {tm['cancer_fn']}")

print("\n--- ROC/PR AUC ---")
print(f"  ROC AUC:            {tm['cancer_roc_auc']:.4f}")
print(f"  PR AUC:             {tm['cancer_pr_auc']:.4f}")

METRICAS MODELO FINAL

--- Metricas de Test ---
  Accuracy:           0.4522
  Balanced Accuracy:  0.3279
  F1-macro:           0.3337
  F1-weighted:        0.4007

--- Deteccion de Cancer (Etapa 1) ---
  Umbral:             0.5766
  Recall (Sens.):     0.8988
  Especificidad:      0.6690
  Precision:          0.8592
  FNR:                0.1012
  TP: 293, FP: 48, TN: 97, FN: 33

--- ROC/PR AUC ---
  ROC AUC:            0.8756
  PR AUC:             0.9350


## Comparacion con notebook 5.0 (18 tipos de cancer originales)

In [ ]:
# Resultados de referencia del notebook 5.0 (mejor modelo)
ref_50 = {
    "cancer_fn": 33,
    "cancer_fnr": 0.1012,
    "cancer_recall": 0.8988,
    "cancer_specificity": 0.6690,
    "cancer_precision": 0.8592,
    "f1_macro": 0.3636,
    "accuracy": 0.5520,
    "balanced_accuracy": 0.3483,
    "cancer_roc_auc": 0.8756,
    "cancer_pr_auc": 0.9350,
}

tm = final_result["test_metrics"]
res_51 = {
    "cancer_fn": tm["cancer_fn"],
    "cancer_fnr": round(tm["cancer_fnr"], 4),
    "cancer_recall": round(tm["cancer_recall_sensitivity"], 4),
    "cancer_specificity": round(tm["cancer_specificity"], 4),
    "cancer_precision": round(tm["cancer_precision"], 4),
    "f1_macro": round(tm["f1_macro"], 4),
    "accuracy": round(tm["accuracy"], 4),
    "balanced_accuracy": round(tm["balanced_accuracy"], 4),
    "cancer_roc_auc": round(tm["cancer_roc_auc"], 4),
    "cancer_pr_auc": round(tm["cancer_pr_auc"], 4),
}

comparison = pd.DataFrame({
    "5.0 (18 tipos cancer)": ref_50,
    "5.1 (clusters malignos)": res_51,
})
comparison["delta"] = comparison["5.1 (clusters malignos)"] - comparison["5.0 (18 tipos cancer)"]

display(comparison)